# 3 — Get Medoids for the Dynamic Prompt

**Goal:** for each KMeans cluster, identify the **medoid** — the population note that is
most representative of the cluster — and freeze it as the single dynamic-prompt example
for that cluster.

**Rationale:** the `dynamic` strategy conditions the model on the most representative
note of the incoming note's cluster. The medoid is the natural choice: it is a *real*
note (unlike the centroid, which is only a point in embedding space) and it maximises
similarity to the cluster's typical content.

**Contamination control:** medoid candidates exclude every note belonging to a patient
present in the evaluation sample — not merely the 32 sampled notes themselves. Patients
with several admissions carry near-identical admission medication lists (copy-forward),
so a sibling note of an evaluation note would leak its answer just as effectively. This
is the same argument that motivates the one-note-per-patient constraint in the sampling
notebook.

**Inputs** (all produced by `2_get_sample.ipynb`):

| Artefact | Role |
|---|---|
| `embeddings.npy` + `embeddings_note_ids.npy` | L2-normalised embeddings and their id vector |
| `final_dataset.parquet` | population with the `cluster` column |
| `sample.parquet` | the 32 evaluation notes, used only for exclusion |
| `clustering/kmeans_centroids.npy` | the fitted centroids that define the partition |
| `clustering/clustering_metadata.json` | K, embedding model, seed |

## 1. Imports and configuration

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from clinical_notes_extraction.config import PROJECT_ROOT

Paths mirror the constants of `2_get_sample.ipynb` so that the two notebooks write into
the same tree. The clustering artefacts live under `clustering/`; the annotated medoid
table is what the extraction run will load.

In [ ]:
DATA_PATH            = Path(PROJECT_ROOT) / "scripts/3_information_extraction/3_2_medications_on_admission/data"
CLUSTERING_PATH      = DATA_PATH / "clustering"
DYNAMIC_PROMPTS_PATH = DATA_PATH / "annotations/dynamic_prompts"

# Inputs
EMB_CACHE       = DATA_PATH / "embeddings.npy"
EMB_IDS_CACHE   = DATA_PATH / "embeddings_note_ids.npy"
POPULATION_FILE = DATA_PATH / "final_dataset.parquet"
SAMPLE_FILE     = DATA_PATH / "sample.parquet"
CENTROIDS_PATH  = CLUSTERING_PATH / "kmeans_centroids.npy"
METADATA_PATH   = CLUSTERING_PATH / "clustering_metadata.json"

# Outputs
MEDOIDS_TO_ANNOTATE_PATH = CLUSTERING_PATH / "medoids_to_annotate.parquet"   # shortlist -> manual annotation
MEDOID_ANNOTATIONS_FILE  = DATA_PATH / "annotations/dynamic_prompts/dynamic_prompt_medoids_annotations.json"  # annotations come back here
MEDOIDS_PATH             = DATA_PATH / "annotations/dynamic_prompts/medoids_annotated.parquet"               # final table used at inference

ID_COLUMN      = "note_id"
PATIENT_COLUMN = "subject_id"

# Column carried as the few-shot example text. "text" is the full clinical note;
# "meds_on_admission_cleaned" is the medication section only, which costs far fewer
# tokens of the 8192-token context. See section 6 for the token budget check.
MEDOID_TEXT_COLUMN = "text"

# "fitted"      -> use kmeans.cluster_centers_ (the centroids that define .predict())
# "member_mean" -> recompute the mean of the eligible members of each cluster
# These differ: MiniBatchKMeans updates centroids incrementally, so cluster_centers_
# is NOT exactly the mean of the final assignments. See section 4.
CENTROID_SOURCE = "fitted"

# Create the folder structure; no-op if it already exists.
os.makedirs(CLUSTERING_PATH, exist_ok=True)
os.makedirs(DYNAMIC_PROMPTS_PATH, exist_ok=True)

In [ ]:
with open(METADATA_PATH) as f:
    clustering_metadata = json.load(f)

K_CHOSEN = clustering_metadata["k_chosen"]

print(f"Embedding model: {clustering_metadata['embedding_model']} "
      f"(dim {clustering_metadata['embedding_dim']})")
print(f"K = {K_CHOSEN} | corpus at clustering time: {clustering_metadata['n_notes']:,} notes")
print(f"Embeddings normalised at encode time: {clustering_metadata['normalize_embeddings']}")

## 2. Load embeddings and population, joined by `note_id`

The embeddings were encoded with `normalize_embeddings=True`, so `embeddings.npy` is
already the L2-normalised matrix that KMeans was fitted on. No re-normalisation and no
PCA transform is needed here — the medoid search runs in exactly the representation that
produced the partition.

**The join is by id, never by position.** A length assertion cannot detect a permutation:
a parquet rewritten by a different engine, or a `final_dataset.parquet` produced before
the empty-note filter, would silently pair row *i* of the population with the embedding
of a different note and corrupt every medoid. `embeddings_note_ids.npy` exists precisely
to make that failure impossible, so we route every lookup through it.

In [ ]:
embeddings = np.load(EMB_CACHE)
emb_ids    = np.load(EMB_IDS_CACHE, allow_pickle=True)
population = pd.read_parquet(POPULATION_FILE)

assert len(embeddings) == len(emb_ids), "Embedding matrix and id vector disagree in length"
assert pd.Index(emb_ids).is_unique, "Duplicate note_id in the embedding id vector"

# note_id -> row position in the embedding matrix
row_of = pd.Series(np.arange(len(emb_ids)), index=emb_ids)

missing = set(population[ID_COLUMN]) - set(row_of.index)
if missing:
    raise ValueError(
        f"{len(missing)} population note(s) have no embedding, e.g. {sorted(missing)[:5]}. "
        "The population parquet and the embedding cache come from different runs — "
        "re-embed or regenerate final_dataset.parquet before continuing."
    )

print(f"Population: {len(population):,} notes | embeddings: {embeddings.shape}")

### 2.1. Exclude the evaluation sample — at patient level

We drop every note whose `subject_id` appears in the sample, not only the 32 sampled
`note_id`s. The `note_id` clause below is redundant (a sampled note necessarily belongs
to a sampled patient) and is kept only to make the intent explicit.

The cost is negligible — a few dozen notes out of the corpus — but the guarantee is what
lets the dissertation state that no evaluation answer was ever visible in a prompt.

In [ ]:
sample = pd.read_parquet(SAMPLE_FILE)

mask = (
    ~population[ID_COLUMN].isin(sample[ID_COLUMN])
    & ~population[PATIENT_COLUMN].isin(sample[PATIENT_COLUMN])
)

eligible = population[mask].reset_index(drop=True)

# Row positions of the eligible notes in the embedding matrix, resolved by id.
eligible_rows = row_of.loc[eligible[ID_COLUMN]].to_numpy()
eligible_emb  = embeddings[eligible_rows]

assert len(eligible) == len(eligible_emb)
assert eligible[ID_COLUMN].is_unique, "Duplicate note_id in the eligible population"
assert eligible["cluster"].nunique() == K_CHOSEN, (
    "At least one cluster has no eligible candidate after the patient-level exclusion"
)

print(
    f"{len(population):,} population notes | "
    f"{len(sample)} sample notes ({sample[PATIENT_COLUMN].nunique()} patients) excluded | "
    f"{len(population) - len(eligible):,} notes removed | "
    f"{len(eligible):,} medoid candidates"
)

### 2.2. Alignment probe

A cheap one-off check that the vector we attribute to a note really is that note's own
embedding. It re-resolves one id through `row_of` and compares against the raw matrix.

In [ ]:
_probe_id  = eligible[ID_COLUMN].iloc[0]
_probe_vec = embeddings[row_of.loc[_probe_id]]

assert np.allclose(eligible_emb[0], _probe_vec)
assert np.allclose(np.linalg.norm(eligible_emb, axis=1), 1.0, atol=1e-4), (
    "Embeddings are not unit-length — cosine similarity via dot product would be wrong"
)

print("Alignment and normalisation checks passed.")

## 3. Which centroid?

The two options are genuinely different objects and the choice must be documented:

- **`fitted`** — `kmeans.cluster_centers_`, saved as `kmeans_centroids.npy`. These are the
  vectors that `kmeans.predict()` compares an incoming note against, so a medoid chosen
  against them is "the most typical member according to the model's own definition of the
  cluster". Note that MiniBatchKMeans updates centroids incrementally from mini-batches,
  so they are *not* exactly the mean of the final assignments.
- **`member_mean`** — the mean of this cluster's eligible members. Under cosine similarity
  this yields the **exact medoid**, not an approximation: since
  `Σⱼ cos(x, yⱼ) = x · Σⱼ yⱼ = n · (x · c)`, maximising similarity to the mean *is*
  maximising the summed similarity to every member. It also has the property of being
  computed on the same population the candidates are drawn from.

`fitted` is the default because it keeps selection and assignment on the same criterion.
Both are computed below so the divergence can be reported.

In [ ]:
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    # Scale each vector to unit length; the clip guards degenerate all-zero rows.
    # Normalising a centroid is a division by a positive scalar, so it never changes
    # the argmax within a cluster - it only makes the resulting similarities true
    # cosines, and therefore comparable across clusters.
    norms = np.linalg.norm(matrix, axis=-1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)


fitted_centroids = l2_normalize(np.load(CENTROIDS_PATH))
assert fitted_centroids.shape == (K_CHOSEN, eligible_emb.shape[1])

# Mean of the eligible members, per cluster, in cluster-label order.
member_mean_centroids = l2_normalize(
    np.vstack([
        eligible_emb[eligible.index[eligible["cluster"] == c].to_numpy()].mean(axis=0)
        for c in range(K_CHOSEN)
    ])
)

drift = (fitted_centroids * member_mean_centroids).sum(axis=1)
print("Cosine between fitted centroid and member mean, per cluster:")
for c, d in enumerate(drift):
    print(f"  cluster {c}: {d:.4f}")

## 4. Find the medoid of each cluster

For each cluster we take the cosine similarity of every eligible member to the chosen
centroid and keep the most central note. Because the embeddings are unit vectors, the
dot product *is* the cosine — one matrix product per cluster, linear in the number of
members.

In [ ]:
centroids = fitted_centroids if CENTROID_SOURCE == "fitted" else member_mean_centroids
print(f"Selecting medoids against the '{CENTROID_SOURCE}' centroids.")

records = []

for cluster_id, group in eligible.groupby("cluster"):
    # group.index are true row positions: `eligible` was reset and `eligible_emb`
    # was built from the same id lookup, in the same order.
    member_idx = group.index.to_numpy()

    # One dot product per member: (n_members x dim) @ (dim,) -> (n_members,)
    similarities = eligible_emb[member_idx] @ centroids[int(cluster_id)]

    best = int(np.argmax(similarities))
    row  = eligible.loc[member_idx[best]]

    records.append({
        "cluster": int(cluster_id),
        ID_COLUMN: row[ID_COLUMN],
        PATIENT_COLUMN: row[PATIENT_COLUMN],
        "similarity_to_centroid": float(similarities[best]),
        "n_members": len(member_idx),
        "margin_over_runner_up": float(
            np.sort(similarities)[-1] - np.sort(similarities)[-2]
        ) if len(similarities) > 1 else float("nan"),
    })

medoid_summary = pd.DataFrame(records).sort_values("cluster").reset_index(drop=True)

assert len(medoid_summary) == K_CHOSEN, f"Expected {K_CHOSEN} medoids, got {len(medoid_summary)}"
assert medoid_summary[ID_COLUMN].is_unique

medoid_summary

`similarity_to_centroid` is how representative the medoid is of its cluster; a low value
signals a diffuse cluster whose example is a weak proxy for its members, which is worth
reporting in the dissertation alongside the cluster sizes. `margin_over_runner_up` shows
how arbitrary the pick was — a near-zero margin means several notes were equally central,
which is not a problem but does mean the specific choice is not meaningful.

## 5. Build the medoid table and export it for manual annotation

In [ ]:
MEDOID_COLUMNS = [
    "note_id", "subject_id", "text", "meds_on_admission_cleaned",
    "meds_on_admission_cleaned_length", "cluster",
    "meds_on_admission_cleaned_length_classification",
    "meds_on_admission_cleaned_length_binary",
]

medoids = (
    eligible[eligible[ID_COLUMN].isin(medoid_summary[ID_COLUMN])][MEDOID_COLUMNS]
    .merge(
        medoid_summary[[ID_COLUMN, "similarity_to_centroid", "n_members"]],
        on=ID_COLUMN,
        how="left",
    )
    .sort_values("cluster")
    .reset_index(drop=True)
)

assert len(medoids) == K_CHOSEN, "isin() returned an unexpected number of rows"

medoids.to_parquet(MEDOIDS_TO_ANNOTATE_PATH)
print(f"Shortlist for manual annotation saved to {MEDOIDS_TO_ANNOTATE_PATH}")

medoids[["cluster", "note_id", "similarity_to_centroid", "n_members",
         "meds_on_admission_cleaned_length"]]

### 5.1. Manual annotation — stop here on a first run

Annotate these notes by hand using the **same schema as the ground truth** (verbatim
`span` per medication), and write the result to `MEDOID_ANNOTATIONS_FILE` as a list of
objects keyed by `note_id`:

```json
[
  {"note_id": 12345678, "medications": [{"name": "...", "span": "...", "...": "..."}]},
  ...
]
```

An empty `medications` list is not acceptable for a note that genuinely lists
medications — the example would teach the model to return nothing. The validation below
fails loudly rather than letting a blank example reach the extraction run.

## 6. Attach the annotations

In [ ]:
if not MEDOID_ANNOTATIONS_FILE.exists():
    raise FileNotFoundError(
        f"No annotation file at {MEDOID_ANNOTATIONS_FILE}. Annotate the "
        f"{K_CHOSEN} notes in {MEDOIDS_TO_ANNOTATE_PATH.name} first (section 5.1)."
    )

with open(MEDOID_ANNOTATIONS_FILE) as f:
    raw_annotations = json.load(f)

annotation_map = {a[ID_COLUMN]: a for a in raw_annotations}

missing_annotations = set(medoids[ID_COLUMN]) - set(annotation_map)
extra_annotations   = set(annotation_map) - set(medoids[ID_COLUMN])

if missing_annotations:
    raise ValueError(f"Medoids without annotation: {sorted(missing_annotations)}")
if extra_annotations:
    print(f"WARNING: annotations for notes that are no longer medoids: {sorted(extra_annotations)}")

print(f"Loaded {len(annotation_map)} annotations.")

Two validations before the annotations are frozen into the artefact:

1. **No empty example.** Every medoid must carry at least one medication.
2. **Spans are verbatim.** Each `span` must occur literally in the note text — a span the
   annotator paraphrased or retyped would teach the model an extraction target that does
   not exist in the input.

In [ ]:
problems = []

for _, row in medoids.iterrows():
    annotation = annotation_map[row[ID_COLUMN]]
    medications = annotation.get("medications", [])

    if not medications:
        problems.append(f"note {row[ID_COLUMN]} (cluster {row['cluster']}): empty medication list")
        continue

    for med in medications:
        span = med.get("span")
        if span and span not in row["text"]:
            problems.append(
                f"note {row[ID_COLUMN]}: span not found verbatim in the note -> {span[:60]!r}"
            )

if problems:
    raise ValueError("Annotation validation failed:\n  " + "\n  ".join(problems))

print(f"All {len(medoids)} annotations validated: non-empty, spans verbatim.")

In [ ]:
# Stored as a JSON string, not a nested dict. pyarrow would otherwise try to infer a
# struct type from the medication objects and either fail or coerce fields when an
# optional key is absent from some annotations. The consumer does json.loads().
# Continua a ser guardado como string JSON (não dict aninhado): agora com mais chaves
# de topo e opcionais, a inferência de struct do pyarrow falharia ainda mais facilmente.
medoids["annotation_json"] = medoids[ID_COLUMN].map(
    lambda nid: json.dumps(annotation_map[nid], ensure_ascii=False)
)

medoids[["cluster", "note_id", "annotation_json"]].head()

### 6.1. Context budget

`MEDOID_TEXT_COLUMN = "text"` carries the full clinical note as the few-shot example.
With `num_ctx=8192`, a full discharge note as the example *plus* the note being extracted
can exceed the window — and Ollama truncates silently, with no error. The rough character
count below is a smoke test; switch `MEDOID_TEXT_COLUMN` to `meds_on_admission_cleaned`
if the examples are too heavy.

In [ ]:
CHARS_PER_TOKEN = 4          # crude but adequate for a budget smoke test
CONTEXT_TOKENS  = 8192

budget = pd.DataFrame({
    "cluster": medoids["cluster"],
    "example_chars": medoids[MEDOID_TEXT_COLUMN].str.len(),
    "annotation_chars": medoids["annotation_json"].str.len(),
})
budget["approx_tokens"] = (
    (budget["example_chars"] + budget["annotation_chars"]) / CHARS_PER_TOKEN
).round().astype(int)
budget["share_of_context"] = (budget["approx_tokens"] / CONTEXT_TOKENS).round(2)

if (budget["share_of_context"] > 0.4).any():
    print("WARNING: at least one example takes more than 40% of the context window, "
          "leaving little room for the note being extracted.")

budget

## 7. Attach the embedding and save

The `dynamic` strategy ranks the frozen examples against the embedding of the incoming
note, so the vector travels with the table and the extraction run never loads the
embedding model. The vector stored is the **L2-normalised** one, identical to the matrix
used for clustering — so at inference the ranking is a plain dot product, with no
normalisation step to forget on either side.

Note that parquet stores these as object arrays: `np.vstack(df["embedding"].to_numpy())`
rebuilds the `(K x dim)` matrix on read.

In [ ]:
medoid_rows = row_of.loc[medoids[ID_COLUMN]].to_numpy()
medoids["embedding"] = list(embeddings[medoid_rows].astype(np.float32))

# The vector attached to each row must be the one the medoid was selected with.
for i, row in medoids.iterrows():
    assert np.allclose(row["embedding"], embeddings[row_of.loc[row[ID_COLUMN]]])

final_medoids = medoids[["note_id", "cluster", MEDOID_TEXT_COLUMN, "annotation_json",
                         "similarity_to_centroid", "embedding"]].rename(
    columns={MEDOID_TEXT_COLUMN: "text"}
).sort_values("cluster").reset_index(drop=True)

final_medoids

In [ ]:
final_medoids.to_parquet(MEDOIDS_PATH)
print(f"Saved {len(final_medoids)} medoids to {MEDOIDS_PATH}")

# Provenance: what these medoids were selected from and how.
provenance = {
    "k": K_CHOSEN,
    "centroid_source": CENTROID_SOURCE,
    "text_column": MEDOID_TEXT_COLUMN,
    "population_notes": int(len(population)),
    "eligible_candidates": int(len(eligible)),
    "excluded_notes": int(len(population) - len(eligible)),
    "excluded_patients": int(sample[PATIENT_COLUMN].nunique()),
    "exclusion_rule": "patient-level: every note of a patient present in the evaluation sample",
    "embedding_model": clustering_metadata["embedding_model"],
}

with open(CLUSTERING_PATH / "medoids_metadata.json", "w") as f:
    json.dump(provenance, f, indent=2)

provenance

### 7.1. Read-back check

Reloading the artefact and re-validating it catches parquet round-trip surprises (dtype
coercion on the embedding column, a JSON string that no longer parses) here rather than
in the middle of the extraction grid.

In [ ]:
check = pd.read_parquet(MEDOIDS_PATH)

vectors = np.vstack(check["embedding"].to_numpy())
assert vectors.shape == (K_CHOSEN, embeddings.shape[1])
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-4)
assert sorted(check["cluster"]) == list(range(K_CHOSEN))
assert all(json.loads(a) for a in check["annotation_json"])
assert not set(check["note_id"]) & set(sample[ID_COLUMN])

print("Read-back OK: vectors unit-length, one medoid per cluster, "
      "annotations parse, no overlap with the evaluation sample.")